# 1103 43일차

## 1. train_test_split과 K-Fold 함께 쓰기

전체를 train과 test로 먼저 나눈 뒤, train 안에서만 K-Fold를 돌리는 방식 (`m03_kfold_train_test_split00_iris`, `m03_kfold_train_test_split01_california`)

K-Fold만 쓰면 모든 데이터가 검증에 참여해서 한 번도 보지 않은 데이터로 최종 확인할 몫이 남지 않음. test를 먼저 떼어 두면 모델 선택은 train 안의 K-Fold로 하고, 고른 모델을 test로 한 번 확인할 수 있음

1. `train_test_split` : 전체를 train(80%)과 test(20%)로 나눔
2. `scaler.fit_transform(x_train)` / `scaler.transform(x_test)` : 스케일링
3. `KFold(n_splits=5, shuffle=True, random_state=333)` : train을 나눌 fold 설정
4. `cross_val_score(model, x_train, y_train, cv=kfold)` : train만 넣어서 교차 검증

```python
x_train, x_test, y_train, y_test = train_test_split(
    x, y, shuffle=True, random_state=333, train_size=0.8, stratify=y,
)
kfold = KFold(n_splits=5, shuffle=True, random_state=333)
scores = cross_val_score(model, x_train, y_train, cv=kfold)
```

- 스케일러는 train에만 `fit_transform`, test에는 `transform`만 씀. test에 `fit`하면 test의 분포가 학습에 섞임
- `cross_val_score`에 `x_train`을 넣으므로 fold의 크기는 train의 1/5. 전체 기준으로는 16%
- 분류는 `stratify=y`로 라벨 비율을 맞추고, 회귀는 y가 연속값이라 생략
- fold별 점수가 서로 비슷하게 나오면 데이터가 고르게 나뉘었다는 뜻
- `n_split` 같은 변수로 빼 두면 fold 개수를 한 곳에서 바꿀 수 있음

## 2. cross_val_predict

교차 검증을 돌리면서 나온 예측값을 모아서 돌려주는 함수 (`m03_kfold_train_test_split00_iris`, `m03_kfold_train_test_split07_santander`)

`cross_val_score`는 점수만 돌려줘서 다른 지표로 다시 계산할 수 없음. 예측값을 받아 두면 원하는 평가 지표를 직접 골라 쓸 수 있음

1. `cross_val_predict(model, x, y, cv=kfold)` : fold마다 나온 예측값을 원래 순서대로 모아서 돌려줌
2. `accuracy_score(y_test, y_predict)` / `r2_score(y_test, y_predict)` : 받은 예측값으로 지표 계산

```python
y_predict = cross_val_predict(model, x_test, y_test, cv=kfold)
print(accuracy_score(y_test, y_predict))
```

- `cross_val_score`는 점수 배열, `cross_val_predict`는 예측값 배열을 돌려줌
- 예측값과 실제값을 나란히 출력하면 어느 샘플을 틀렸는지 눈으로 확인할 수 있음
- 분류는 `accuracy_score`, 회귀는 `r2_score`로 계산

#### cf) cross_val_predict 점수가 더 낮은 이유

- 넘긴 데이터를 다시 k등분해서, 각 조각을 나머지 조각으로 학습한 모델이 예측함
- `x_test`만 넘기면 전체의 20%를 또 5등분하므로, 학습에 쓰이는 양이 크게 줄어 점수가 낮게 나옴
- `cross_val_score`의 평균과 숫자가 다르다고 모델이 나빠진 것이 아니고, 학습에 쓴 데이터 양이 달라서 생기는 차이

## 3. all_estimators

sklearn에 들어 있는 모델을 한꺼번에 불러와 차례로 돌려 보는 기능 (`m04_all_estimators1_classifier`, `m04_all_estimators2_regressor`)

데이터에 어떤 모델이 맞는지 미리 알 수 없어서, 후보를 하나씩 바꿔 가며 쓰는 대신 전부 돌려 점수를 비교함

1. `all_estimators(type_filter='classifier')` : `(이름, 클래스)` 튜플 목록을 돌려줌
2. `model = model()` : 목록에서 받은 것은 클래스라서 괄호를 붙여 객체로 만듦
3. `model.fit(x_train, y_train)` → `model.score(x_test, y_test)` : 모델마다 학습과 평가
4. `try` / `except` : 중간에 에러가 나도 멈추지 않고 다음 모델로 넘어가게 감쌈

```python
all = all_estimators(type_filter='classifier')
for name, model in all:
    try:
        model = model()
        model.fit(x_train, y_train)
        print(name, model.score(x_test, y_test))
    except:
        print(name, '모델 오류')
```

- `type_filter`는 `'classifier'`와 `'regressor'`로 분류·회귀 모델을 나눠서 받음
- 모델 개수는 sklearn 버전에 따라 다르고 `sk.__version__`으로 확인
- 생성할 때 인자가 더 필요한 모델이나 데이터 형태가 맞지 않는 모델은 에러가 나므로 `try`/`except`가 필요
- `warnings.filterwarnings('ignore')` : 모델마다 쏟아지는 경고를 숨겨서 점수만 보이게 함
- `cross_val_score`와 같이 쓰면 모델마다 K-Fold 평균 점수를 비교할 수 있음 (`m05_all_estimators1_kfold`, `m05_all_estimators2_kfold`)